In [1]:
# importing the packages
import numpy as np
from ortools.linear_solver import pywraplp
from warnings import filterwarnings

filterwarnings("ignore")

In [3]:
def read_data(index=0):
    # reading the cost_matrix and the pair_matrix
    cost_matrix = np.loadtxt("../data/cost_matrix/cost.txt").reshape(-1,1)
    pair_matrix = np.genfromtxt("../data/pairings/pair_array.txt", delimiter=',', dtype='int')
    if index != 0:
        cost_matrix = cost_matrix[index]
        pair_matrix = pair_matrix[index]
    # determining the number of flights and tasks
    num_pairs = pair_matrix.shape[0]
    num_flights = pair_matrix.shape[1]
    print(num_pairs, num_flights)
    
    return cost_matrix, pair_matrix, num_pairs, num_flights

index = [866, 4274, 7795, 11481, 12867, 14253, 17113, 19474, 22259, 24417, 25903, 28182]
cost_matrix, pair_matrix, num_pairs, num_flights = read_data()

80706 240


In [58]:
# Initializing the LP Solver
solver = pywraplp.Solver.CreateSolver("GLOP")

# creating the binary allocation variable
x = np.array([solver.NumVar(0, 1, f"x_{i}") for i in range (num_pairs)]).reshape(-1,1)

# create a matrix that is the product of the decision variable and the pair matrix
result_matrix = x * pair_matrix

# declaring the constraints
# Uniqueness of flight legs and all flight legs covered
for i in range(num_flights):
    solver.Add(solver.Sum(result_matrix[:,i]) >= 1)
    
# declaring the objective function
solver.Minimize(np.sum(cost_matrix * x))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
if status == pywraplp.Solver.OPTIMAL:
    print("The Solution is OPTIMAL")
elif status == pywraplp.Solver.FEASIBLE:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

x_values = [x[i][0].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

# gives the optimal value faster but does not give the optimal pairs correctly
# gives the dual values correctly
# reads sub optimal values correctly and gives the expected values

The Solution is OPTIMAL
80706
[1624, 3421, 3656, 3945, 4306, 6146, 7094, 9642, 9855, 9955, 10150, 11507, 11531, 11815, 13079, 13741, 14487, 14693, 15837, 15887, 15978, 16103, 16104, 16199, 16212, 18032, 19781, 19815, 21924, 21953, 21959, 22138, 22591, 23403, 23405, 23940, 25049, 25381, 26741, 26775, 27599, 27637, 28032, 28214, 28980, 29061, 29128, 29145, 29173, 29181, 29216, 29221, 29240, 29357, 32156, 32653, 32663, 32853, 34258, 34988, 35409, 39710, 40049, 40197, 41940, 42109, 42110, 42225, 42316, 42650, 43040, 43406, 43438, 45267, 45480, 49328, 49376, 49633, 50327, 51240, 52210, 52558, 52625, 52667, 53045, 53520, 53638, 54369, 54371, 56472, 56682, 57403, 57522, 59263, 60681, 60772, 61745, 61762, 61778, 61871, 61963, 62307, 62323, 64200, 64467, 64484, 65020, 65184, 65901, 66457, 66689, 67894, 67933, 68364, 68431, 69388, 69406, 69467, 72156, 72173, 72182, 72306, 72324, 72379, 72424, 72458, 72485, 72550, 72741, 72931, 73188, 73557, 74169, 74560, 74573, 74773, 74873, 75088, 75122, 75144,

In [55]:
# Initializing the MIP Solver
solver = pywraplp.Solver.CreateSolver("SAT")

# creating the binary allocation variable
x = np.array([solver.BoolVar("") for i in range (num_pairs)]).reshape(-1,1)

# create a matrix that is the product of the decision variable and the pair matrix
result_matrix = x * pair_matrix

# declaring the constraints
# Uniqueness of flight legs and all flight legs covered
for i in range(num_flights):
    solver.Add(solver.Sum(result_matrix[:,i]) >= 1)

# declaring the objective function
solver.Minimize(np.sum(cost_matrix * x))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
if status == pywraplp.Solver.OPTIMAL:
    print("The Solution is OPTIMAL")
elif status == pywraplp.Solver.FEASIBLE:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

x_values = [x[i][0].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

The Solution is OPTIMAL
32978
[866, 4274, 7795, 11481, 12867, 14253, 17113, 19474, 22259, 24417, 25903, 28182]
157.0
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [52]:
#SAT Solver
# Initializing the MIP Solver
solver = pywraplp.Solver.CreateSolver("SAT")

# Decision Variable
x = [solver.IntVar(0.0, 1.0, f'x_{i}') for i in range (num_pairs)]

# Constraints
for j in range(num_flights):
    solver.Add(sum(pair_matrix[i][j] * x[i] for i in range(num_pairs)) -1 >= 0)

# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
if status == pywraplp.Solver.OPTIMAL:
    print("The Solution is OPTIMAL")
elif status == pywraplp.Solver.FEASIBLE:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

x_values = [x[i].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

The Solution is OPTIMAL
12
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
157.0
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [35]:
# importing the packages
import numpy as np
from ortools.linear_solver import pywraplp

# Initialize the solver
solver = pywraplp.Solver.CreateSolver("CBC")
use_dual_simplex: True

# Decision Variable
x = [solver.NumVar(0, 1, f'x_{i}') for i in range(num_pairs)]

# Constraints
for j in range(num_flights):
    solver.Add(sum(pair_matrix[i][j] * x[i] for i in range(num_pairs)) -1 >= 0)

# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
x_values = [x[i].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

157.0
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
32978
[559, 814, 3241, 3883, 4055, 4230, 7495, 8917, 11736, 12096, 12389, 12423, 12510, 12794, 12907, 14296, 16640, 19094, 19213, 19483, 19543, 19917, 20119, 21563, 24417, 24451, 26595, 26621, 28197, 28365]
157.0
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [36]:
# importing the packages
import numpy as np
from ortools.sat.python import cp_model

# Declare the CP_SAT model
model = cp_model.CpModel()

# create the decision variable
x = []
for i in range(num_pairs):
    x.append(model.NewBoolVar(f'x_{i}'))

# Constraints
constraints = []
for j in range(num_flights):
    constraints.append(model.Add(sum(pair_matrix[i][j] * x[i] for i in range(num_pairs)) >= 1))

# Objective Function
model.Minimize(sum(cost_matrix[i][0] * x[i] for i in range(num_pairs)))

# Solve the Problem
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL:
    selected_pairs = [i for i in range(num_pairs) if solver.Value(x[i]) == 1]
    print(selected_pairs)
else:
    print("oops")

print(solver.ObjectiveValue(), len(selected_pairs))

[866, 4274, 7795, 11481, 12867, 14253, 17113, 19474, 22259, 24417, 25903, 28182]
157.0 12


In [4]:
import numpy as np
from scipy.optimize import linprog

def solve_mip(pair_matrix, cost_matrix):
    num_pairs, num_flights = pair_matrix.shape
    
    # Objective coefficients for the binary decision variables
    c = cost_matrix.flatten().reshape(-1,1)
    print(c.shape)
    # Coefficients matrix for the constraints (each flight leg should be covered at least once)
    A_eq = np.vstack([pair_matrix, np.ones((1, num_flights))])
    b_eq = np.ones(num_flights + 1)  # RHS for equality constraints

    # Bounds for decision variables (x should be binary)
    bounds = [(0, 1) for _ in range(num_pairs)]
    print(A_eq.shape)
    # Solve the linear programming problem
    result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

    # Extract the results
    minimized_cost = result.fun
    selected_pairs = np.round(result.x).astype(int)
    dual_variables = result.slack[:-1]  # Slack variables correspond to dual variables in equality constraints

    return minimized_cost, selected_pairs, dual_variables


minimized_cost, selected_pairs, dual_variables = solve_mip(pair_matrix, cost_matrix)

# Print the results
print("Minimized Cost:", minimized_cost)
print("Selected Pairs:", selected_pairs)
print("Optimal Dual Variables:", dual_variables)


(80706, 1)
(80707, 240)


ValueError: Invalid input for linprog: A_eq must have exactly two dimensions, and the number of columns in A_eq must be equal to the size of c